# Fretwork — Simple Inference (v2: transformer decoder)

**Audio file in Drive → MIDI notes + guitar tab**, using the best benchmarked configuration: Basic Pitch (tuned thresholds) → audio key/chord context → **transformer beam decoder** (`prox_viterbi_transformer`, weight 2.0, beam 8 — 0.782 given-pitch on validation).

**How to use:** set `AUDIO_FILE` in the config cell, then Runtime → Run all. Outputs print below and save to `Capstone/outputs/simple_inference/`.

**One-time prerequisite (already done if the file exists):** the decoder's candidate costs include the GuitarSet unigram prior. Export it once from `AudioToTab_VariantEval_v1` (any session where `POSITION_PRIOR_COSTS` exists) by running:

```python
import json
with open(CAPSTONE_ROOT / 'guitarset_position_prior.json', 'w') as f:
    json.dump({f'{m},{s},{fr}': c for (m, s, fr), c in POSITION_PRIOR_COSTS.items()}, f)
```

If the file is missing, this notebook still runs — the unigram term becomes a constant and drops out — but fingering may differ slightly from the benchmarked configuration. The trained transformer (`Capstone/transformer_prior/tab_transformer_final.pt`) must exist — run the export cell in `Transformer_Position_Prior.ipynb` once if it does not. A GPU runtime is recommended but not required; on CPU a 30–60s clip decodes in a minute or two.


In [1]:
# ============================================================
# FIX-ALL Basic Pitch install/import cell for Colab Python 3.12
# Run this ONCE after Runtime -> Restart runtime
# ============================================================

import sys
import subprocess
import importlib
import pkgutil
import zipimport

print("Python:", sys.version)

def run(cmd):
    print("\n$", " ".join(cmd))
    subprocess.check_call(cmd)

# ------------------------------------------------------------
# 1. Patch Python 3.12 pkg_resources issue BEFORE imports
# ------------------------------------------------------------
# Some Colab/system pkg_resources versions expect pkgutil.ImpImporter,
# which was removed in Python 3.12.
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = zipimport.zipimporter

# ------------------------------------------------------------
# 2. Keep setuptools modern enough for Python 3.12,
#    but below torch's <82 constraint
# ------------------------------------------------------------
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "wheel", "setuptools==80.9.0"])

# ------------------------------------------------------------
# 3. Install Basic Pitch dependencies manually
#    This avoids pip backtracking into old basic-pitch/numpy versions.
# ------------------------------------------------------------
run([sys.executable, "-m", "pip", "install", "-q",
     "librosa>=0.10",
     "soundfile",
     "mir-eval",
     "pretty_midi",
     "resampy==0.4.2",
     "onnxruntime"])

# ------------------------------------------------------------
# 4. Force Basic Pitch latest without dependency resolver chaos
# ------------------------------------------------------------
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "basic-pitch==0.4.0"])

# ------------------------------------------------------------
# 5. Import test
# ------------------------------------------------------------
# Re-apply patch right before import in case anything reset it.
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = zipimport.zipimporter

import librosa
import soundfile as sf
from basic_pitch.inference import predict as basic_pitch_predict

print("\n✅ Basic Pitch import successful.")
print("✅ librosa:", librosa.__version__)
print("✅ soundfile import successful.")
print("✅ onnxruntime installed.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

$ /usr/bin/python3 -m pip install -q --upgrade pip wheel setuptools==80.9.0

$ /usr/bin/python3 -m pip install -q librosa>=0.10 soundfile mir-eval pretty_midi resampy==0.4.2 onnxruntime

$ /usr/bin/python3 -m pip install -q --no-deps basic-pitch==0.4.0


/usr/local/lib/python3.12/dist-packages/resampy/filters.py:50: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



✅ Basic Pitch import successful.
✅ librosa: 0.11.0
✅ soundfile import successful.
✅ onnxruntime installed.


In [74]:
# ── CONFIG: set your audio file here ─────────────────────────────────────────
import sys
from pathlib import Path
import json, gzip, math, re
from collections import defaultdict, Counter
from itertools import product
import numpy as np
import pandas as pd

# Check if running in Google Colab environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive as _gdrive
    _gdrive.mount('/content/drive', force_remount=False)
    CAPSTONE_ROOT = Path('/content/drive/MyDrive/Capstone')
else:
    NOTEBOOK_DIR = Path.cwd().resolve()
    if NOTEBOOK_DIR.name == 'jupyter_notebooks':
        CAPSTONE_ROOT = NOTEBOOK_DIR.parent
    else:
        CAPSTONE_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'backend').exists() else NOTEBOOK_DIR.parent

AUDIO_FILE = CAPSTONE_ROOT / 'Audio' / 'ode.mp3'   # <-- CHANGE ME

# transformer decoder settings live in the decoder cell below (weight 2.0, beam 8)
GUITARSET_PRIOR_PATH = CAPSTONE_ROOT / 'guitarset_position_prior.json'

AUDIO_OUTPUT_DIR = CAPSTONE_ROOT / 'outputs' / 'simple_inference'
AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# fretboard constants (identical to the eval notebook)
MAX_FRET = 24
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]  # E2, A2, D3, G3, B3, E4
STRING_NAMES = ['low_E', 'A', 'D', 'G', 'B', 'high_E']
ONSET_TOLERANCE_SECONDS = 0.035
COMFORTABLE_SPAN = 5
MAX_REACHABLE_SPAN = 7
LARGE_JUMP_THRESHOLD = 5
MAX_GROUP_CANDIDATES = 25

# tuned Basic Pitch settings (identical to the eval notebook)
BASIC_PITCH_AMPLITUDE_THRESHOLD = 0.40
BASIC_PITCH_ONSET_THRESHOLD = 0.50
BASIC_PITCH_FRAME_THRESHOLD = 0.20
BASIC_PITCH_MIN_MIDI = 40
BASIC_PITCH_MAX_MIDI = 88
BASIC_PITCH_CACHE_VERSION = 'tuned_v1'

print('Audio file:', AUDIO_FILE, '| exists:', AUDIO_FILE.exists())
print('Transformer weights:', (CAPSTONE_ROOT / 'transformer_prior' / 'tab_transformer_final.pt').exists() or (CAPSTONE_ROOT / 'tab_transformer_final.pt').exists(),
      '| GuitarSet prior:', GUITARSET_PRIOR_PATH.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Audio file: /content/drive/MyDrive/Capstone/Audio/ode.mp3 | exists: True
Transformer weights: True | GuitarSet prior: False


In [75]:
# ── Fretboard (identical to eval notebook) ────────────────────────────────────
def build_fretboard(open_string_midi=OPEN_STRING_MIDI, max_fret=MAX_FRET):
    rows = []
    for string_idx, open_midi in enumerate(open_string_midi):
        for fret in range(max_fret + 1):
            midi = open_midi + fret
            rows.append({'string': string_idx, 'string_name': STRING_NAMES[string_idx],
                         'fret': fret, 'midi': midi, 'pitch_class': midi % 12})
    return pd.DataFrame(rows)

fretboard_df = build_fretboard()
MIDI_TO_POSITIONS = defaultdict(list)
for row in fretboard_df.to_dict('records'):
    MIDI_TO_POSITIONS[int(row['midi'])].append({
        'string': int(row['string']), 'string_name': row['string_name'],
        'fret': int(row['fret']), 'midi': int(row['midi']), 'pitch_class': int(row['pitch_class'])})

def get_possible_positions(midi_note, max_fret=MAX_FRET):
    midi_note = int(round(midi_note))
    return [p for p in MIDI_TO_POSITIONS.get(midi_note, []) if 0 <= p['fret'] <= max_fret]

print('Fretboard ready:', len(fretboard_df), 'positions')


Fretboard ready: 150 positions


In [76]:
PITCH_CLASS_NAMES_SHARP = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
NOTE_TO_PC = {name: i for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}
NOTE_TO_PC.update({'Db': 1, 'Eb': 3, 'Gb': 6, 'Ab': 8, 'Bb': 10})
PC_TO_NOTE = {i: name for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}

MAJOR_STEPS = [2, 2, 1, 2, 2, 2, 1]
MINOR_STEPS = [2, 1, 2, 2, 1, 2, 2]
MAJOR_QUALITIES = ['maj', 'min', 'min', 'maj', 'maj', 'min', 'dim']
MINOR_QUALITIES = ['min', 'dim', 'maj', 'min', 'min', 'maj', 'maj']

def derive_scale(root_pc, mode='major'):
    steps = MAJOR_STEPS if mode == 'major' else MINOR_STEPS
    pcs = [root_pc]
    cur = root_pc
    for step in steps[:-1]:
        cur = (cur + step) % 12
        pcs.append(cur)
    return pcs

def build_key_database():
    rows = []
    for root_name, root_pc in NOTE_TO_PC.items():
        if 'b' in root_name:
            continue
        for mode in ['major', 'minor']:
            scale_pcs = derive_scale(root_pc, mode)
            qualities = MAJOR_QUALITIES if mode == 'major' else MINOR_QUALITIES
            chords = []
            for degree, (pc, qual) in enumerate(zip(scale_pcs, qualities), start=1):
                chords.append({
                    'degree': degree,
                    'root_pc': pc,
                    'root': PC_TO_NOTE[pc],
                    'quality': qual,
                    'symbol': f'{PC_TO_NOTE[pc]}:{qual}',
                })
            rows.append({
                'key': f'{root_name} {mode}',
                'root': root_name,
                'root_pc': root_pc,
                'mode': mode,
                'scale_pcs': scale_pcs,
                'scale_notes': [PC_TO_NOTE[pc] for pc in scale_pcs],
                'diatonic_chords': chords,
            })
    return pd.DataFrame(rows)

key_db = build_key_database()


In [77]:
CHORD_INTERVALS = {
    'maj': [0, 4, 7],
    'min': [0, 3, 7],
    'dim': [0, 3, 6],
    'aug': [0, 4, 8],
    '7': [0, 4, 7, 10],
    'maj7': [0, 4, 7, 11],
    'min7': [0, 3, 7, 10],
    'm7': [0, 3, 7, 10],
    'sus4': [0, 5, 7],
    'sus2': [0, 2, 7],
    '5': [0, 7],
}
QUALITY_ALIASES = {'M': 'maj', 'major': 'maj', '': 'maj', 'm': 'min', 'minor': 'min', 'dom7': '7'}

def normalize_quality(q):
    if q is None:
        return 'maj'
    q = str(q).strip()
    return QUALITY_ALIASES.get(q, q)

def chord_tones(root_pc, quality='maj'):
    quality = normalize_quality(quality)
    intervals = CHORD_INTERVALS.get(quality, CHORD_INTERVALS['maj'])
    return sorted({(root_pc + i) % 12 for i in intervals})

def parse_chord_symbol(symbol):
    if symbol is None:
        return None
    s = str(symbol).strip()
    # Remove inversion/bass-note suffixes such as D:7/1 or C:maj/G before parsing quality.
    s = s.split('/')[0]
    if s in ['N', 'X', 'nan', 'None', '']:
        return None
    if ':' in s:
        root, qual = s.split(':', 1)
    else:
        m = re.match(r'^([A-G](?:#|b)?)(.*)$', s)
        if not m:
            return None
        root, qual = m.group(1), m.group(2)
    if root not in NOTE_TO_PC:
        return None
    qual = normalize_quality(qual)
    return {'root': root, 'root_pc': NOTE_TO_PC[root], 'quality': qual, 'tones': chord_tones(NOTE_TO_PC[root], qual)}

def recognize_chord_from_pitches(midi_pitches, allowed_qualities=('maj', 'min', 'dim', '7', 'maj7', 'min7')):
    pcs = sorted({int(round(m)) % 12 for m in midi_pitches})
    if not pcs:
        return None
    best = None
    for root_pc in range(12):
        for qual in allowed_qualities:
            tones = set(chord_tones(root_pc, qual))
            pcs_set = set(pcs)
            precision = len(pcs_set & tones) / max(len(pcs_set), 1)
            recall = len(pcs_set & tones) / max(len(tones), 1)
            score = 2 * precision * recall / (precision + recall + 1e-9)
            cand = {'symbol': f'{PC_TO_NOTE[root_pc]}:{qual}', 'root_pc': root_pc, 'quality': qual, 'tones': sorted(tones), 'score': score}
            if best is None or cand['score'] > best['score']:
                best = cand
    return best


In [78]:
def infer_key_from_filename(recording_name):
    parts = recording_name.split('_')
    if len(parts) >= 2:
        middle = parts[1]
        key_guess = middle.split('-')[-1]
        if key_guess in NOTE_TO_PC:
            return f'{key_guess} major'
    return None

def get_key_info(key_label):
    if key_label is None:
        return None
    s = str(key_label).replace(':', ' ').strip()
    # Normalize GuitarSet-style labels such as D:major into D major.
    toks = s.split()
    if len(toks) == 1 and toks[0] in NOTE_TO_PC:
        s = f'{toks[0]} major'
    match = key_db[key_db['key'] == s]
    return match.iloc[0].to_dict() if len(match) else None

def chord_end_time(c):
    # Some parsed chord dictionaries have duration but not an explicit end time.
    # This helper keeps the rest of the notebook robust either way.
    start = float(c.get('start', 0.0))
    if c.get('end') is not None:
        return float(c['end'])
    return start + float(c.get('duration', 0.0) or 0.0)

def chord_at_time(chords, t):
    for c in chords:
        start = float(c.get('start', 0.0))
        end = chord_end_time(c)
        if start <= t < end:
            return c
    return None

def enrich_notes_with_context(record):
    key_label = record.get('key') or infer_key_from_filename(record['recording'])
    key_info = get_key_info(key_label)
    out = []
    for n in record['notes']:
        c = chord_at_time(record['chords'], n['start'])
        row = dict(n)
        row['key_label'] = key_label
        row['in_key'] = None if key_info is None else (n['pitch_class'] in set(key_info['scale_pcs']))
        row['chord_label'] = None if c is None else c['chord']
        parsed_chord = None if c is None else c.get('parsed') or parse_chord_symbol(c.get('chord'))
        row['in_chord'] = None if parsed_chord is None else (n['pitch_class'] in set(parsed_chord['tones']))
        out.append(row)
    return out


In [79]:
def estimate_hand_position_from_frets(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if not fretted else int(round(np.median(fretted)))

def group_span(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if len(fretted) <= 1 else max(fretted) - min(fretted)

def awkward_fingering_penalty(position, hand_center):
    fret = position['fret']
    if fret == 0:
        return 0.0
    distance = abs(fret - hand_center)
    if distance <= 2:
        return 0.0
    if distance <= COMFORTABLE_SPAN:
        return 0.5 * (distance - 2)
    if distance <= MAX_REACHABLE_SPAN:
        return 2.0 + (distance - COMFORTABLE_SPAN)
    return 10.0 + 2.0 * (distance - MAX_REACHABLE_SPAN)

def group_playability_cost(group_positions):
    if not group_positions:
        return 0.0
    strings = [p['string'] for p in group_positions]
    frets = [p['fret'] for p in group_positions]
    fretted = [f for f in frets if f > 0]
    cost = 0.0
    if len(strings) != len(set(strings)):
        return float('inf')
    span = group_span(frets)
    if span > COMFORTABLE_SPAN:
        cost += 2.0 * (span - COMFORTABLE_SPAN)
    if span > MAX_REACHABLE_SPAN:
        cost += 25.0 * (span - MAX_REACHABLE_SPAN)
    if fretted and min(fretted) <= 2 and max(fretted) >= 9:
        cost += 8.0
    if len(strings) >= 2:
        string_span = max(strings) - min(strings)
        if string_span > 4 and len(strings) <= 3:
            cost += 1.5 * (string_span - 4)
    hand_center = estimate_hand_position_from_frets(frets)
    cost += sum(awkward_fingering_penalty(p, hand_center) for p in group_positions)
    if any(f == 0 for f in frets) and fretted and max(fretted) > 7:
        cost += 3.0
    return cost

def transition_cost(prev_group, curr_group):
    if prev_group is None or curr_group is None:
        return 0.0
    prev_frets = [p['fret'] for p in prev_group]
    curr_frets = [p['fret'] for p in curr_group]
    prev_strings = [p['string'] for p in prev_group]
    curr_strings = [p['string'] for p in curr_group]
    prev_center = estimate_hand_position_from_frets(prev_frets)
    curr_center = estimate_hand_position_from_frets(curr_frets)
    cost = 1.2 * abs(curr_center - prev_center) + 0.25 * abs(np.mean(curr_strings) - np.mean(prev_strings))
    if len(prev_group) == 1 and len(curr_group) == 1:
        pf, cf = prev_group[0]['fret'], curr_group[0]['fret']
        ps, cs = prev_group[0]['string'], curr_group[0]['string']
        cost += 0.8 * abs(cf - pf) + 0.35 * abs(cs - ps)
        if abs(cf - pf) > LARGE_JUMP_THRESHOLD:
            cost += 4.0 + abs(cf - pf) - LARGE_JUMP_THRESHOLD
        if cf == 0 and pf > 7:
            cost += 2.0
    return cost

def context_cost(group_notes, group_positions):
    cost = 0.0
    for n, p in zip(group_notes, group_positions):
        if n.get('in_chord') is False:
            cost += 0.15
        if n.get('in_key') is False:
            cost += 0.10
    return cost


In [80]:
def group_notes_by_onset(notes, tolerance=ONSET_TOLERANCE_SECONDS):
    if not notes:
        return []
    notes_sorted = sorted(notes, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))
    groups, current = [], [notes_sorted[0]]
    group_start = notes_sorted[0]['start']
    for n in notes_sorted[1:]:
        if abs(n['start'] - group_start) <= tolerance:
            current.append(n)
        else:
            groups.append(current)
            current = [n]
            group_start = n['start']
    groups.append(current)
    return groups

def enrich_candidate(candidate):
    positions = candidate['positions']
    frets = [p['fret'] for p in positions]
    strings = [p['string'] for p in positions]
    candidate['center'] = estimate_hand_position_from_frets(frets)
    candidate['avg_string'] = float(np.mean(strings)) if strings else 0.0
    candidate['is_single'] = len(positions) == 1
    candidate['single_fret'] = positions[0]['fret'] if len(positions) == 1 else np.nan
    candidate['single_string'] = positions[0]['string'] if len(positions) == 1 else np.nan
    return candidate

def candidate_groups_for_notes(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)
    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue
        base_cost = group_playability_cost(combo) + context_cost(group_notes, combo)
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    if not candidates:
        for combo in product(*position_lists):
            combo = list(combo)
            base_cost = group_playability_cost(combo)
            if math.isinf(base_cost):
                base_cost = 1000.0
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


In [81]:
# ── GuitarSet unigram prior (exported once from the eval notebook) ───────────
POSITION_PRIOR_COSTS = {}
if GUITARSET_PRIOR_PATH.exists():
    with open(GUITARSET_PRIOR_PATH) as f:
        POSITION_PRIOR_COSTS = {tuple(int(x) for x in k.split(',')): float(v)
                                for k, v in json.load(f).items()}
    print(f'GuitarSet unigram prior loaded: {len(POSITION_PRIOR_COSTS)} entries')
else:
    print('WARNING: guitarset_position_prior.json not found - the unigram term becomes a')
    print('constant and drops out. Output may differ slightly from the benchmarked config.')
    print('Export it from AudioToTab_VariantEval_v1 (see the instructions cell above).')

def position_prior_cost(midi, position):
    key = (int(midi), int(position['string']), int(position['fret']))
    return float(POSITION_PRIOR_COSTS.get(key, 0.75))


constant and drops out. Output may differ slightly from the benchmarked config.
Export it from AudioToTab_VariantEval_v1 (see the instructions cell above).


In [82]:
# ============================================================
# CAGED-box algorithm, wired into THIS eval harness.
# Reuses existing get_possible_positions / context_cost / group_notes_by_onset /
# enrich_candidate / awkward_fingering_penalty etc. so metrics stay comparable.
# Separate names (..._caged) so combined_all_tuned is left untouched.
# ============================================================
COMFORTABLE_CHORD_SPAN, MAX_CHORD_SPAN = 3, 4   # FIX 1: chord span hard wall at 4

def group_playability_cost_caged(gp):
    if not gp: return 0.0
    strings=[p['string'] for p in gp]; frets=[p['fret'] for p in gp]
    fretted=[f for f in frets if f>0]
    if len(strings)!=len(set(strings)): return float('inf')
    cost=0.0; span=group_span(frets)
    if span>COMFORTABLE_CHORD_SPAN: cost+=2.0*(span-COMFORTABLE_CHORD_SPAN)
    if span>MAX_CHORD_SPAN:        cost+=25.0*(span-MAX_CHORD_SPAN)
    if fretted and min(fretted)<=2 and max(fretted)>=9: cost+=8.0
    if len(strings)>=2:
        ss=max(strings)-min(strings)
        if ss>4 and len(strings)<=3: cost+=1.5*(ss-4)
    hc=estimate_hand_position_from_frets(frets)
    cost+=sum(awkward_fingering_penalty(p,hc) for p in gp)
    if any(f==0 for f in frets) and fretted and max(fretted)>7: cost+=3.0
    return cost

BOX_WINDOW, SHIFT_FREE = 4, 2
BOX_CENTER_COST, BOX_OUTSIDE_COST, OPEN_OUT_OF_BOX_COST = 0.15, 3.00, 0.60
BOX_OFFBOX_COST, BOX_NONHOME_COST, BOX_LOWNECK_COST = 0.60, 1.00, 0.04
CAGED_WEIGHTS={'playability':0.80,'context':0.30,'box_window':1.00,'hand_move':0.70,'position_prior':1.00}
PENTATONIC={'major':[0,2,4,7,9],'minor':[0,3,5,7,10]}
LOW_E_PC=OPEN_STRING_MIDI[0]%12

def parse_key(key_label):
    info=get_key_info(key_label)
    return None if info is None else {'root_pc':int(info['root_pc']),'mode':info['mode'],'scale_pcs':set(info['scale_pcs'])}

def box_anchors_for_key(key,max_fret=MAX_FRET,window=BOX_WINDOW):
    rng=range(0,max_fret-window+1)
    if key is None:
        return [{'anchor':a,'key_cost':BOX_LOWNECK_COST*a} for a in rng]
    r=key['root_pc']; penta=PENTATONIC.get(key['mode'],PENTATONIC['minor'])
    box=set()
    for deg in penta:
        f=(deg+(r-LOW_E_PC))%12
        while f<=max_fret-1: box.add(f); f+=12
    home=set(); h=(r-LOW_E_PC)%12
    while h<=max_fret-1: home.add(h); h+=12
    out=[]
    for a in rng:
        d=min((abs(a-b) for b in box),default=0)
        kc=BOX_OFFBOX_COST*d+(0.0 if a in home else BOX_NONHOME_COST)+BOX_LOWNECK_COST*a
        out.append({'anchor':a,'key_cost':kc})
    return out

def position_window_cost(p,anchor,window=BOX_WINDOW):
    f=p['fret']
    if f==0: return 0.0 if anchor<=2 else OPEN_OUT_OF_BOX_COST
    if anchor<=f<=anchor+window: return BOX_CENTER_COST*abs(f-(anchor+window/2.0))
    return BOX_OUTSIDE_COST*((anchor-f) if f<anchor else (f-(anchor+window)))

def candidate_window_cost(c,anchor): return sum(position_window_cost(p,anchor) for p in c['positions'])

def candidate_groups_caged(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    pls=[]
    for n in group_notes:
        pos=get_possible_positions(n['midi'])
        if not pos: return []
        pls.append(pos)

    def _greedy(cost):   # best-effort: one distinct string per note, lowest fret, keeps note order
        used=set(); pick={}
        for i in sorted(range(len(group_notes)), key=lambda k:-group_notes[k]['midi']):
            opts=sorted(pls[i], key=lambda p:(p['fret'],p['string']))
            chosen=next((p for p in opts if p['string'] not in used), opts[0])
            used.add(chosen['string']); pick[i]=chosen
        return enrich_candidate({'positions':[pick[i] for i in range(len(group_notes))],'base_cost':cost})

    space=1
    for pl in pls: space*=len(pl)
    if space>20000:                       # too many simultaneous-note combos -> skip full product
        return [_greedy(10.0)]

    cands=[]
    for combo in product(*pls):
        combo=list(combo)
        if len(combo)>1 and len({p['string'] for p in combo})!=len(combo): continue
        play=group_playability_cost_caged(combo)
        if not math.isfinite(play): continue
        prior=float(np.mean([position_prior_cost(n['midi'],p) for n,p in zip(group_notes,combo)]))
        base=(CAGED_WEIGHTS['playability']*play
              +CAGED_WEIGHTS['context']*context_cost(group_notes,combo)
              +CAGED_WEIGHTS['position_prior']*prior)
        cands.append(enrich_candidate({'positions':combo,'base_cost':float(base)}))

    if not cands:                         # no conflict-free shape -> keep the record alive
        cands.append(_greedy(100.0))

    return sorted(cands,key=lambda c:c['base_cost'])[:max_candidates]

SOLO_MOVE_SCALE = 0.35   # how much to relax hand-position locking between single notes (solos roam)
SOLO_BOX_SCALE  = 0.30   # how much to relax the home-box pull on single notes (let the prior place them)


In [83]:
# ============================================================
# Chord-voicing library: reward candidate placements that form a known hand shape.
# Plugs a bonus into the existing caged candidate cost. Only affects chord onsets.
# ============================================================
VOICING_SHAPES = [   # string idx 0=lowE..5=highE; fret offsets relative to lowest fretted note
    {'name':'E-maj', 'offsets':{0:0,1:2,2:2,3:1,4:0,5:0}, 'power':False},
    {'name':'A-maj', 'offsets':{1:0,2:2,3:2,4:2,5:0},     'power':False},
    {'name':'D-maj', 'offsets':{2:0,3:2,4:3,5:2},         'power':False},
    {'name':'C-maj', 'offsets':{1:3,2:2,3:0,4:1,5:0},     'power':False},
    {'name':'G-maj', 'offsets':{0:3,1:2,2:0,3:0,4:0,5:3}, 'power':False},
    {'name':'E-min', 'offsets':{0:0,1:2,2:2,3:0,4:0,5:0}, 'power':False},
    {'name':'A-min', 'offsets':{1:0,2:2,3:2,4:1,5:0},     'power':False},
    {'name':'E-7',   'offsets':{0:0,1:2,2:0,3:1,4:0,5:0}, 'power':False},
    {'name':'A-7',   'offsets':{1:0,2:2,3:0,4:2,5:0},     'power':False},
    {'name':'E-m7',  'offsets':{0:0,1:2,2:0,3:0,4:0,5:0}, 'power':False},
    {'name':'A-m7',  'offsets':{1:0,2:2,3:0,4:1,5:0},     'power':False},
    {'name':'Emaj7', 'offsets':{0:0,1:2,2:1,3:1,4:0,5:0}, 'power':False},
    {'name':'Amaj7', 'offsets':{1:0,2:2,3:1,4:2,5:0},     'power':False},
    {'name':'5-E',   'offsets':{0:0,1:2},       'power':True},
    {'name':'5-A',   'offsets':{1:0,2:2},       'power':True},
    {'name':'5-D',   'offsets':{2:0,3:2},       'power':True},
    {'name':'5-E-oct','offsets':{0:0,1:2,2:2},  'power':True},
    {'name':'5-A-oct','offsets':{1:0,2:2,3:2},  'power':True},
    {'name':'5-D-oct','offsets':{2:0,3:2,4:2},  'power':True},
    {'name':'oct-E', 'offsets':{0:0,2:2},       'power':True},
    {'name':'oct-A', 'offsets':{1:0,3:2},       'power':True},
]

def _off_from_min(d):
    m = min(d.values()); return {k: v - m for k, v in d.items()}

def voicing_bonus(positions):
    """0.0 if the placement isn't a recognized shape; 0.6..1.0 if it is (higher = fuller match).
    Transposition-invariant; matches partial chords; 2-note groups only match power/octave shapes."""
    pts = {p['string']: p['fret'] for p in positions}
    strings = sorted(pts)
    if len(strings) < 2: return 0.0
    cand_off = _off_from_min(pts); n = len(strings); best = 0.0
    for sh in VOICING_SHAPES:
        if (n < 2) if sh['power'] else (n < 3): continue
        smap = sh['offsets']
        if not all(s in smap for s in strings): continue
        if _off_from_min({s: smap[s] for s in strings}) == cand_off:
            best = max(best, 0.6 + 0.4 * (n / len(smap)))
    return best

CAGED_WEIGHTS['voicing'] = 1.0   # tune on validation; try 0.5–2.0

def candidate_groups_voiced(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    # pull a wider pool so the bonus can promote a shape the base ranking would have truncated
    pool = candidate_groups_caged(group_notes, max_candidates=max(max_candidates * 3, 24))
    w = CAGED_WEIGHTS.get('voicing', 0.0)
    if w and len(group_notes) >= 2:
        for c in pool:
            b = voicing_bonus(c['positions'])
            if b: c['base_cost'] = c['base_cost'] - w * b
        pool = sorted(pool, key=lambda c: c['base_cost'])
    return pool[:max_candidates]


In [84]:
BASIC_PITCH_CACHE_DIR = AUDIO_OUTPUT_DIR / 'basic_pitch_note_cache'
BASIC_PITCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Default context toggles if they were not defined in the config cell.
# By default, this is a full audio-driven pipeline: no GT key/chords as input.
try:
    USE_GROUND_TRUTH_KEY_FOR_CONTEXT
except NameError:
    USE_GROUND_TRUTH_KEY_FOR_CONTEXT = False
try:
    USE_GROUND_TRUTH_CHORDS_FOR_CONTEXT
except NameError:
    USE_GROUND_TRUTH_CHORDS_FOR_CONTEXT = False
try:
    USE_AUDIO_KEY_DETECTION
except NameError:
    USE_AUDIO_KEY_DETECTION = True
try:
    USE_AUDIO_CHORD_DETECTION
except NameError:
    USE_AUDIO_CHORD_DETECTION = True
try:
    CHORD_WINDOW_SECONDS
except NameError:
    CHORD_WINDOW_SECONDS = 1.0
try:
    CHORD_HOP_SECONDS
except NameError:
    CHORD_HOP_SECONDS = 0.5
try:
    MIN_NOTES_PER_CHORD_WINDOW
except NameError:
    MIN_NOTES_PER_CHORD_WINDOW = 2

def midi_to_note_name_simple(midi):
    names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    midi = int(round(midi))
    return f"{names[midi % 12]}{midi // 12 - 1}"

def _basic_pitch_cache_path(audio_path):
    audio_path = Path(audio_path)
    safe_name = audio_path.stem.replace('/', '_')
    return BASIC_PITCH_CACHE_DIR / f'{safe_name}_{BASIC_PITCH_CACHE_VERSION}_bp_notes.csv'

def run_basic_pitch_notes(audio_path,
                          amplitude_threshold=BASIC_PITCH_AMPLITUDE_THRESHOLD,
                          onset_threshold=BASIC_PITCH_ONSET_THRESHOLD,
                          frame_threshold=BASIC_PITCH_FRAME_THRESHOLD,
                          min_midi=BASIC_PITCH_MIN_MIDI,
                          max_midi=BASIC_PITCH_MAX_MIDI,
                          use_cache=True):
    """Run tuned Basic Pitch and return notes for the fretboard algorithm."""
    audio_path = Path(audio_path)
    cache_path = _basic_pitch_cache_path(audio_path)

    if use_cache and cache_path.exists():
        df = pd.read_csv(cache_path)
        return df.to_dict('records')

    print(f'Running Basic Pitch on: {audio_path.name}')
    _, _, note_events = basic_pitch_predict(
        str(audio_path),
        onset_threshold=onset_threshold,
        frame_threshold=frame_threshold,
    )

    notes = []
    for event in note_events:
        # Basic Pitch usually returns: start, end, pitch_midi, amplitude, bends
        start, end, pitch_midi, amplitude = event[0], event[1], event[2], event[3]
        if float(amplitude) < amplitude_threshold:
            continue
        midi = int(round(float(pitch_midi)))
        if midi < min_midi or midi > max_midi:
            continue
        notes.append({
            'start': float(start),
            'duration': float(end - start),
            'midi': midi,
            'pitch_class': midi % 12,
            'note_name': midi_to_note_name_simple(midi),
            'amplitude': float(amplitude),
            # Ground truth unknown for audio-derived predictions.
            'true_string': None,
            'true_fret': None,
            'source': 'basic_pitch',
        })

    notes = sorted(notes, key=lambda n: (n['start'], n['midi']))
    pd.DataFrame(notes).to_csv(cache_path, index=False)
    return notes

# Krumhansl-Schmuckler key profiles for simple audio key estimation.
_MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
_MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

def detect_key_from_audio(audio_path):
    y, sr = librosa.load(str(audio_path), sr=None, mono=True)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    chroma_avg = np.nan_to_num(np.mean(chroma, axis=1))
    scores = []
    for tonic in range(12):
        major_key = f'{PC_TO_NOTE[tonic]} major'
        minor_key = f'{PC_TO_NOTE[tonic]} minor'
        major_score = np.corrcoef(chroma_avg, np.roll(_MAJOR_PROFILE, tonic))[0, 1]
        minor_score = np.corrcoef(chroma_avg, np.roll(_MINOR_PROFILE, tonic))[0, 1]
        scores.append((major_key, major_score))
        scores.append((minor_key, minor_score))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return {'key': scores[0][0], 'score': float(scores[0][1]), 'top5': scores[:5]}

# Lightweight chord templates. These are inferred from Basic Pitch notes, not JAMS chords.
_CHORD_TEMPLATES = []
for root in range(12):
    _CHORD_TEMPLATES.extend([
        {'label': PC_TO_NOTE[root],       'root': root, 'quality': 'maj',  'tones': {(root + x) % 12 for x in [0, 4, 7]}},
        {'label': PC_TO_NOTE[root] + 'm', 'root': root, 'quality': 'min',  'tones': {(root + x) % 12 for x in [0, 3, 7]}},
        {'label': PC_TO_NOTE[root] + '7', 'root': root, 'quality': 'dom7', 'tones': {(root + x) % 12 for x in [0, 4, 7, 10]}},
    ])

def best_chord_for_pitch_classes(pitch_classes):
    """Return the best lightweight chord template for a set/list of pitch classes.

    This keeps dictionary objects OUT of the comparison tuple. Otherwise,
    Python can crash on ties with:
    TypeError: '>' not supported between instances of 'dict' and 'dict'.
    """
    pcs = set(int(pc) % 12 for pc in pitch_classes)
    if len(pcs) < 2:
        return None

    best_score_tuple = None
    best_template = None

    for templ_idx, templ in enumerate(_CHORD_TEMPLATES):
        tones = templ['tones']
        overlap = len(pcs & tones)
        missing = len(tones - pcs)
        extra = len(pcs - tones)
        root_bonus = 0.35 if templ['root'] in pcs else 0.0
        score = overlap - 0.45 * missing - 0.25 * extra + root_bonus

        # Compare only numeric values. The final -templ_idx is a deterministic tie-breaker.
        score_tuple = (score, overlap, -missing, -extra, -templ_idx)

        if best_score_tuple is None or score_tuple > best_score_tuple:
            best_score_tuple = score_tuple
            best_template = templ

    if best_template is None or best_score_tuple[1] < 2:
        return None

    return best_template

def detect_chords_from_basic_pitch_notes(notes, window_seconds=CHORD_WINDOW_SECONDS, hop_seconds=CHORD_HOP_SECONDS):
    """Detect rough chord context from Basic Pitch note pitch classes in sliding windows."""
    if not notes:
        return []
    max_time = max(float(n['start']) + float(n.get('duration', 0.0) or 0.0) for n in notes)
    chords = []
    t = 0.0
    current = None
    prev_label = None

    while t <= max_time:
        t_end = t + window_seconds
        pcs = []
        for n in notes:
            n_start = float(n['start'])
            n_end = n_start + float(n.get('duration', 0.0) or 0.0)
            if n_start < t_end and n_end >= t:
                pcs.append(int(n['pitch_class']))

        templ = best_chord_for_pitch_classes(pcs) if len(pcs) >= MIN_NOTES_PER_CHORD_WINDOW else None
        label = None if templ is None else templ['label']

        if label is not None:
            parsed = {'root': templ['root'], 'quality': templ['quality'], 'tones': sorted(templ['tones'])}
            if current is not None and label == prev_label:
                current['end'] = t_end
                current['duration'] = current['end'] - current['start']
            else:
                if current is not None:
                    chords.append(current)
                current = {
                    'start': float(t),
                    'end': float(t_end),
                    'duration': float(window_seconds),
                    'chord': label,
                    'parsed': parsed,
                    'source': 'basic_pitch_window_chords',
                }
                prev_label = label
        else:
            if current is not None:
                chords.append(current)
                current = None
            prev_label = None

        t += hop_seconds

    if current is not None:
        chords.append(current)
    return chords

def make_audio_record_from_gt(record, audio_path):
    """Build a model-input record from audio-derived notes/context. GT is used only later for scoring."""
    bp_notes = run_basic_pitch_notes(audio_path)

    if USE_GROUND_TRUTH_KEY_FOR_CONTEXT:
        key = record.get('key') or infer_key_from_filename(record['recording'])
        key_source = 'ground_truth_jams_or_filename'
    elif USE_AUDIO_KEY_DETECTION:
        try:
            key_pred = detect_key_from_audio(audio_path)
            key = key_pred['key']
            key_source = 'audio_chroma'
        except Exception as e:
            print(f'Audio key detection failed for {Path(audio_path).name}: {e}')
            key = infer_key_from_filename(record['recording'])
            key_source = 'filename_fallback'
    else:
        key = infer_key_from_filename(record['recording'])
        key_source = 'filename_fallback'

    if USE_GROUND_TRUTH_CHORDS_FOR_CONTEXT:
        chords = record.get('chords', [])
        chord_source = 'ground_truth_jams'
    elif USE_AUDIO_CHORD_DETECTION:
        chords = detect_chords_from_basic_pitch_notes(bp_notes)
        chord_source = 'basic_pitch_window_chords' if chords else 'none_detected'
    else:
        chords = []
        chord_source = 'none'

    return {
        'recording': record['recording'],
        'path': str(audio_path),
        'notes': bp_notes,
        'chords': chords,
        'beats': [],
        'tempo': record.get('tempo'),
        'key': key,
        'key_source': key_source,
        'chord_source': chord_source,
    }


In [85]:
# ============================================================
# prox-Viterbi -> BEAM SEARCH with transformer position scoring
# ============================================================
import torch as _torch
import torch.nn as _nn

TRANSFORMER_PRIOR_PATH = CAPSTONE_ROOT / 'transformer_prior' / 'tab_transformer_final.pt'
TRANSFORMER_WEIGHT = 2.0       # best on validation
BEAM_WIDTH = 8

_TP_MIN_MIDI, _TP_N_PITCH, _TP_N_POS = 40, 49, 150
_TP_BOS = 150
_TP_DEVICE = 'cuda' if _torch.cuda.is_available() else 'cpu'

class _TabTransformer(_nn.Module):
    def __init__(self, d, layers, heads, ctx):
        super().__init__()
        self.ctx = ctx
        self.emb_pitch = _nn.Embedding(_TP_N_PITCH, d)
        self.emb_prev  = _nn.Embedding(_TP_N_POS + 1, d)
        self.emb_flag  = _nn.Embedding(2, d)
        self.emb_time  = _nn.Embedding(ctx, d)
        layer = _nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=4*d,
                                            dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = _nn.TransformerEncoder(layer, num_layers=layers)
        self.head = _nn.Linear(d, _TP_N_POS)
    def forward(self, pitch, prev_pos, flag):
        B, T = pitch.shape
        t_idx = _torch.arange(T, device=pitch.device).unsqueeze(0).expand(B, T)
        x = (self.emb_pitch(pitch) + self.emb_prev(prev_pos)
             + self.emb_flag(flag) + self.emb_time(t_idx))
        causal = _torch.triu(_torch.ones(T, T, dtype=_torch.bool, device=pitch.device), 1)
        return self.head(self.encoder(x, mask=causal))

_ck = _torch.load(TRANSFORMER_PRIOR_PATH, map_location=_TP_DEVICE)
_cfg = _ck['config']
TP_MODEL = _TabTransformer(_cfg['d'], _cfg['layers'], _cfg['heads'], _cfg['ctx']).to(_TP_DEVICE)
TP_MODEL.load_state_dict(_ck['model']); TP_MODEL.eval()
_TP_CTX = _cfg['ctx']

_VALID = np.zeros((_TP_N_PITCH, _TP_N_POS), dtype=bool)
for _s, _om in enumerate(OPEN_STRING_MIDI):
    for _f in range(25):
        _m = _om + _f
        if _TP_MIN_MIDI <= _m <= _TP_MIN_MIDI + _TP_N_PITCH - 1:
            _VALID[_m - _TP_MIN_MIDI, _s * 25 + _f] = True
_VALID_T = _torch.tensor(_VALID, dtype=_torch.bool, device=_TP_DEVICE)

def _score_extensions(histories, extensions):
    """histories: list of (pitch_list, pos_list, flag_list) per beam.
    extensions: list of lists - extensions[b] = candidate extensions for beam b,
      each a (pitches, positions, flags) tuple for the new group's notes.
    Returns nll[b][c] = total transformer NLL of extension c under beam b."""
    seq_p, seq_prev, seq_g, meta = [], [], [], []
    for b, (hp, hq, hg) in enumerate(histories):
        for c, (ep, eq, eg) in enumerate(extensions[b]):
            p = (hp + ep)[-_TP_CTX:]
            q = (hq + eq)[-_TP_CTX:]
            g = (hg + eg)[-_TP_CTX:]
            prev = [_TP_BOS] + q[:-1]
            seq_p.append(p); seq_prev.append(prev); seq_g.append(g)
            meta.append((b, c, len(ep), len(p)))
    maxlen = max(len(s) for s in seq_p)
    def pad(seqs, val):
        return _torch.tensor([s + [val] * (maxlen - len(s)) for s in seqs],
                             dtype=_torch.long, device=_TP_DEVICE)
    P, PR, G = pad(seq_p, 0), pad(seq_prev, _TP_BOS), pad(seq_g, 0)
    with _torch.no_grad():
        logits = TP_MODEL(P, PR, G)
        logits = logits.masked_fill(~_VALID_T[P], -1e9)
        logp = _torch.log_softmax(logits, dim=-1)
    out = defaultdict(dict)
    for row, (b, c, n_new, L) in enumerate(meta):
        nll = 0.0
        full_q = (histories[b][1] + [pos for pos in extensions[b][c][1]])[-_TP_CTX:]
        for t in range(L - n_new, L):
            nll -= float(logp[row, t, full_q[t]])
        out[b][c] = nll
    return out

def assign_prox_viterbi_transformer(notes, key=None):
    groups = group_notes_by_onset(notes)
    if not groups:
        return []
    beams = [{'hp': [], 'hq': [], 'hg': [], 'cost': 0.0, 'centers': None, 'choice': []}]
    allc = []
    for g in groups:
        cands = candidate_groups_voiced(g)
        allc.append(cands if cands else None)

    for gi, g in enumerate(groups):
        cands = allc[gi]
        if cands is None:
            for b in beams:
                b['choice'].append(None)
            continue
        g_sorted_idx = sorted(range(len(g)), key=lambda k: g[k]['midi'])
        exts_per_cand = []
        for c in cands:
            pos_sorted = sorted(c['positions'], key=lambda p_: p_['midi'])
            ep = [int(p_['midi']) - _TP_MIN_MIDI for p_ in pos_sorted]
            eq = [int(p_['string']) * 25 + int(p_['fret']) for p_ in pos_sorted]
            eg = [1] + [0] * (len(pos_sorted) - 1)
            exts_per_cand.append((ep, eq, eg))
        histories = [(b['hp'], b['hq'], b['hg']) for b in beams]
        extensions = [exts_per_cand for _ in beams]
        nll = _score_extensions(histories, extensions)

        scored = []
        for bi, b in enumerate(beams):
            for ci, c in enumerate(cands):
                cf = [p_['fret'] for p_ in c['positions'] if p_['fret'] > 0]
                cc = float(np.mean(cf)) if cf else 0.0
                move = 0.0
                if b['centers'] is not None:
                    move = CAGED_WEIGHTS['hand_move'] * abs(cc - b['centers'])
                total = (b['cost'] + c['base_cost'] + move
                         + TRANSFORMER_WEIGHT * nll[bi][ci])
                scored.append((total, bi, ci, cc if cf else b['centers']))
        scored.sort(key=lambda x: x[0])
        new_beams = []
        for total, bi, ci, center in scored[:BEAM_WIDTH]:
            b = beams[bi]; ep, eq, eg = exts_per_cand[ci]
            new_beams.append({
                'hp': (b['hp'] + ep)[-_TP_CTX:], 'hq': (b['hq'] + eq)[-_TP_CTX:],
                'hg': (b['hg'] + eg)[-_TP_CTX:], 'cost': total,
                'centers': center, 'choice': b['choice'] + [ci]})
        beams = new_beams

    best = min(beams, key=lambda b: b['cost'])
    out = []
    for gi, (g, ci) in enumerate(zip(groups, best['choice'])):
        if ci is None or allc[gi] is None:
            continue
        c = allc[gi][ci]
        pos_sorted = sorted(c['positions'], key=lambda p_: p_['midi'])
        g_sorted = sorted(g, key=lambda n_: n_['midi'])
        for note, p_ in zip(g_sorted, pos_sorted):
            row = dict(note)
            row.update({'pred_string': p_['string'], 'pred_fret': p_['fret'],
                        'method': 'prox_viterbi_transformer'})
            out.append(row)
    return sorted(out, key=lambda x: (x['start'], x['midi']))

print(f'prox_viterbi_transformer ready (device={_TP_DEVICE}, beam={BEAM_WIDTH}, W={TRANSFORMER_WEIGHT})')


prox_viterbi_transformer ready (device=cpu, beam=8, W=2.0)


/tmp/ipykernel_1646/2917542836.py:25: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = _nn.TransformerEncoder(layer, num_layers=layers)


In [86]:
# ── Tab renderer (systems of onset columns) ──────────────────────────────────
def render_tab(pred_rows, cols_per_line=24):
    rows_sorted = sorted(pred_rows, key=lambda r: (float(r['start']), int(r['midi'])))
    groups = group_notes_by_onset(rows_sorted)
    systems = []
    for start in range(0, len(groups), cols_per_line):
        chunk = groups[start:start + cols_per_line]
        lines = [[] for _ in range(6)]
        for g in chunk:
            by_string = {int(r['pred_string']): int(r['pred_fret'])
                         for r in g if r.get('pred_string') is not None}
            width = max((len(str(f)) for f in by_string.values()), default=1)
            for s in range(6):
                cell = str(by_string[s]) if s in by_string else '-' * width
                lines[s].append(cell.rjust(width, '-'))
        block = []
        for s in reversed(range(6)):
            block.append(f"{STRING_NAMES[s]:>6}|-" + '-'.join(lines[s]) + '-|')
        systems.append(chr(10).join(block))
    return (chr(10) + chr(10)).join(systems)


In [87]:
# ── RUN: audio -> MIDI notes -> tab ──────────────────────────────────────────
assert AUDIO_FILE.exists(), f'Audio file not found: {AUDIO_FILE}'

stub = {'recording': AUDIO_FILE.stem, 'notes': [], 'chords': [], 'key': None, 'tempo': None}
audio_record = make_audio_record_from_gt(stub, AUDIO_FILE)
notes_ctx = [n for n in enrich_notes_with_context(audio_record) if get_possible_positions(n['midi'])]
print(f"{AUDIO_FILE.name}: {len(notes_ctx)} notes detected | "
      f"key={audio_record.get('key')} ({audio_record.get('key_source')}) | "
      f"{len(audio_record.get('chords', []))} chord windows")

tab_rows = assign_prox_viterbi_transformer(notes_ctx)

# MIDI note table
NOTE_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
notes_df = pd.DataFrame([{
    'start_sec': round(float(r['start']), 3),
    'duration_sec': round(float(r.get('duration', 0.0)), 3),
    'midi': int(r['midi']),
    'note': f"{NOTE_NAMES[int(r['midi']) % 12]}{int(r['midi']) // 12 - 1}",
    'string': STRING_NAMES[int(r['pred_string'])],
    'fret': int(r['pred_fret']),
    'amplitude': round(float(r.get('amplitude', 0.0)), 3),
} for r in tab_rows])
display(notes_df.head(25))

# Tab
tab_text = render_tab(tab_rows)
print()
print(tab_text)

# Save both to Drive
notes_path = AUDIO_OUTPUT_DIR / f'{AUDIO_FILE.stem}_notes.csv'
tab_path   = AUDIO_OUTPUT_DIR / f'{AUDIO_FILE.stem}_tab.txt'
notes_df.to_csv(notes_path, index=False)
tab_path.write_text(tab_text)
print(f'\nSaved: {notes_path}\nSaved: {tab_path}')


ode.mp3: 30 notes detected | key=E minor (audio_chroma) | 17 chord windows


,start_sec,duration_sec,midi,note,string,fret,amplitude
0,1.126,0.557,64,E4,B,5,0.506
1,1.683,0.466,64,E4,B,5,0.593
2,2.196,0.499,65,F4,B,6,0.707
3,2.776,0.546,67,G4,B,8,0.575
4,3.322,0.464,67,G4,B,8,0.612
5,3.879,0.384,65,F4,B,6,0.636
6,4.438,0.464,64,E4,B,5,0.578
7,4.983,0.499,62,D4,G,7,0.720
8,5.552,0.489,60,C4,G,5,0.595
9,6.041,0.430,60,C4,G,5,0.669



high_E|-------------------------------------------------|
     B|-5-5-6-8-8-6-5---------5-5-----5-5-6-8-8-6-5-----|
     G|---------------7-5-5-7-----7-7---------------7-5-|
     D|-------------------------------------------------|
     A|-------------------------------------------------|
 low_E|-------------------------------------------------|

high_E|-------------|
     B|-----5-------|
     G|-5-7---7-5-5-|
     D|-------------|
     A|-------------|
 low_E|-------------|

Saved: /content/drive/MyDrive/Capstone/outputs/simple_inference/ode_notes.csv
Saved: /content/drive/MyDrive/Capstone/outputs/simple_inference/ode_tab.txt


In [88]:
# ── Multiple ways to play it: anchored beam decoding ──────────────────────────
# Runs the transformer beam decoder with a gentle pull toward different neck
# regions, producing distinct, internally-coherent arrangements of the same
# recording. anchor=None reproduces the standard decode.

ANCHOR_PULL = 0.9      # cost per fret of distance from the anchor region (must outweigh stay-put costs)
MIN_DISTINCT_FRAC = 0.25   # variants must differ on at least this fraction of notes

def assign_transformer_anchored(notes, anchor=None):
    """Beam decode with an optional soft anchor toward a neck region."""
    groups = group_notes_by_onset(notes)
    if not groups:
        return []
    beams = [{'hp': [], 'hq': [], 'hg': [], 'cost': 0.0, 'centers': None, 'choice': []}]
    allc = [candidate_groups_voiced(g) or None for g in groups]

    for gi, g in enumerate(groups):
        cands = allc[gi]
        if cands is None:
            for b in beams:
                b['choice'].append(None)
            continue
        exts_per_cand = []
        for c in cands:
            pos_sorted = sorted(c['positions'], key=lambda p_: p_['midi'])
            ep = [int(p_['midi']) - _TP_MIN_MIDI for p_ in pos_sorted]
            eq = [int(p_['string']) * 25 + int(p_['fret']) for p_ in pos_sorted]
            eg = [1] + [0] * (len(pos_sorted) - 1)
            exts_per_cand.append((ep, eq, eg))
        histories = [(b['hp'], b['hq'], b['hg']) for b in beams]
        extensions = [exts_per_cand for _ in beams]
        nll = _score_extensions(histories, extensions)

        scored = []
        for bi, b in enumerate(beams):
            for ci, c in enumerate(cands):
                cf = [p_['fret'] for p_ in c['positions'] if p_['fret'] > 0]
                cc = float(np.mean(cf)) if cf else 0.0
                move = 0.0 if b['centers'] is None else CAGED_WEIGHTS['hand_move'] * abs(cc - b['centers'])
                anchor_cost = 0.0
                if anchor is not None and cf:
                    anchor_cost = ANCHOR_PULL * abs(cc - anchor)
                total = (b['cost'] + c['base_cost'] + move + anchor_cost
                         + TRANSFORMER_WEIGHT * nll[bi][ci])
                scored.append((total, bi, ci, cc if cf else b['centers']))
        scored.sort(key=lambda x: x[0])
        new_beams = []
        for total, bi, ci, center in scored[:BEAM_WIDTH]:
            b = beams[bi]; ep, eq, eg = exts_per_cand[ci]
            new_beams.append({'hp': (b['hp'] + ep)[-_TP_CTX:], 'hq': (b['hq'] + eq)[-_TP_CTX:],
                              'hg': (b['hg'] + eg)[-_TP_CTX:], 'cost': total,
                              'centers': center, 'choice': b['choice'] + [ci]})
        beams = new_beams

    best = min(beams, key=lambda b: b['cost'])
    out = []
    for gi, (g, ci) in enumerate(zip(groups, best['choice'])):
        if ci is None or allc[gi] is None:
            continue
        c = allc[gi][ci]
        pos_sorted = sorted(c['positions'], key=lambda p_: p_['midi'])
        for note, p_ in zip(sorted(g, key=lambda n_: n_['midi']), pos_sorted):
            row = dict(note)
            row.update({'pred_string': p_['string'], 'pred_fret': p_['fret']})
            out.append(row)
    return sorted(out, key=lambda x: (x['start'], x['midi']))

def _variant_fingerprint(rows):
    return [(int(r['pred_string']), int(r['pred_fret'])) for r in rows]

def _distinct_frac(a, b):
    n = min(len(a), len(b))
    if n == 0: return 1.0
    return sum(1 for i in range(n) if a[i] != b[i]) / n

def multiway_tabs(notes, anchors=(None, 3, 8)):
    """Decode several arrangements; keep the distinct ones, labeled by region."""
    variants = []
    for a in anchors:
        rows = assign_transformer_anchored(list(notes), anchor=a)
        if not rows:
            continue
        frets = [r['pred_fret'] for r in rows if r['pred_fret'] > 0]
        med = float(np.median(frets)) if frets else 0.0
        label = ('open position' if med <= 3 else
                 f'around fret {int(round(med))}')
        fp = _variant_fingerprint(rows)
        if any(_distinct_frac(fp, v['fp']) < MIN_DISTINCT_FRAC for v in variants):
            continue                      # near-duplicate of an earlier variant
        variants.append({'anchor': a, 'label': label, 'rows': rows, 'fp': fp})
    return variants

variants = multiway_tabs(notes_ctx)
print(f'{len(variants)} distinct arrangement(s):')
for v in variants:
    print(f"\n=== {v['label']} ===")
    print(render_tab(v['rows']))
    out_path = AUDIO_OUTPUT_DIR / f"{AUDIO_FILE.stem}_tab_{v['label'].replace(' ', '_')}.txt"
    out_path.write_text(render_tab(v['rows']))
print('\nSaved one .txt per arrangement in', AUDIO_OUTPUT_DIR)

3 distinct arrangement(s):

=== around fret 6 ===
high_E|-------------------------------------------------|
     B|-5-5-6-8-8-6-5---------5-5-----5-5-6-8-8-6-5-----|
     G|---------------7-5-5-7-----7-7---------------7-5-|
     D|-------------------------------------------------|
     A|-------------------------------------------------|
 low_E|-------------------------------------------------|

high_E|-------------|
     B|-----5-------|
     G|-5-7---7-5-5-|
     D|-------------|
     A|-------------|
 low_E|-------------|

=== open position ===
high_E|-0-0-1-3-3-1-0---------0-0-----0-0-1-3-3-1-0-----|
     B|---------------3-1-1-3-----3-3---------------3-1-|
     G|-------------------------------------------------|
     D|-------------------------------------------------|
     A|-------------------------------------------------|
 low_E|-------------------------------------------------|

high_E|-----0-------|
     B|-1-3---3-1-1-|
     G|-------------|
     D|-------------|
     A|--